# Step 8 — Robust normal-form curves, and the sampling consequence of h_net

**Fix.** The Step-7 coefficient plots blew up for τ > 1 where the drift is nearly linear (cubic coefficient → 0, so the cube-completing shift diverged). Now: cubic coefficient clamped to `b ≥ 0.02`, shift clipped to |s| ≤ 0.6.

**Prediction to test.** Step 7 showed undertrained networks inject a spurious asymmetry `h_net` (e.g. +0.45 at ep20 vs −0.01 exact for (0,1)), decaying with training. In the cusp picture, `h > 0` favors the branch on the `+` side, so **generated samples should be biased toward that class in proportion to h_net, with the bias shrinking as training proceeds.** We sample 2,000 images per checkpoint with a deterministic Heun sampler and measure the class fraction along the LDA axis.

Checkpoints: Add data → the Step 7 notebook output (it retrained and saved `unet_*_ep*.pt`). GPU on, ~10 min.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, glob, os
import matplotlib.pyplot as plt
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
torch.manual_seed(0); rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 110
ds = datasets.MNIST('/kaggle/working/mnist', train=True, download=True)
X_ALL = (ds.data.numpy().astype(np.float32)/127.5 - 1.0); Y_ALL = ds.targets.numpy()
PAIRS = [(0, 1), (3, 8), (0, 8)]; SIGMA_DATA = 0.5; CKPTS = [20, 40, 60, 80]
found = glob.glob('/kaggle/input/**/unet_*_ep*.pt', recursive=True); assert found, 'mount the Step 7 notebook output as a data source'
CKPT_DIR = os.path.dirname(found[0]); print(CKPT_DIR)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__()
        self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1)
        self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]; return x + self.c2(F.silu(self.n2(h)))

class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__()
        self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1)
        self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1); self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31))
        e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e); return self.out(self.u1(self.up(h2) + h1, e))

def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1); c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))

def load_nets(pair):
    out = {}
    for ep in CKPTS:
        net = UNet().to(dev); net.load_state_dict(torch.load(f'{CKPT_DIR}/unet_{pair[0]}{pair[1]}_ep{ep}.pt', map_location=dev)); out[ep] = net.eval()
    return out
NETS = {pair: load_nets(pair) for pair in PAIRS}

GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True)
    k = np.exp(-GRID**2/(2*tau)); k /= k.sum(); return np.maximum(np.convolve(h, k, mode='same'), 1e-300)
def exact_drift(m0, tau):
    p = smooth_hist(m0, tau); return GRID, np.gradient(np.log(p), GRID), p
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def balanced_flat(pair):
    Xa = X_ALL[Y_ALL == pair[0]].reshape(-1, 784); Xb = X_ALL[Y_ALL == pair[1]].reshape(-1, 784)
    nmin = min(len(Xa), len(Xb)); return Xa, Xb, np.concatenate([Xa[:nmin], Xb[:nmin]])

@torch.no_grad()
def net_drift(net, Xflat, u, mid, half, tau, n=30000, nbins=70, smooth_bins=1.5):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); sigma = half*np.sqrt(tau)
    x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
    D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
    m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
    lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/smooth_bins)**2)
    return 0.5*(edges[:-1]+edges[1:]), np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9), cnt

def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    """Robust version: f = h + a*y - b*y^3, y = m - s. Cubic coefficient clamped, shift clipped."""
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok]))
    b = max(-c3, b_min); c3 = -b
    s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    a = c1 + 3*c3*s**2 + 2*c2*s; h = c0 + c1*s + c2*s**2 + c3*s**3
    return dict(h=h, a=a, b=b, s=s, jump_pred=3*(abs(h)/(2*b))**(1/3))

## 1. Normal-form coefficients vs τ and training (readable this time)

In [ ]:
TAU_FIT = np.geomspace(0.3, 2.0, 16); COEF = {}
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for j, pair in enumerate(PAIRS):
    Xa, Xb, Xflat = balanced_flat(pair); u, mid, half = lda_direction(Xa, Xb); m0 = (Xflat-mid) @ u / half
    ex = [cusp_fit(*exact_drift(m0, t)) for t in TAU_FIT]
    for r, key in enumerate(['a', 'h', 'b']): axes[r, j].plot(TAU_FIT, [e[key] for e in ex], 'k-', lw=2.5, label='exact')
    COEF[(pair, 'exact')] = ex
    for ep, col in zip(CKPTS, ['C0', 'C1', 'C2', 'C3']):
        fits = [cusp_fit(*net_drift(NETS[pair][ep], Xflat, u, mid, half, t, n=20000)) for t in TAU_FIT]; COEF[(pair, ep)] = fits
        for r, key in enumerate(['a', 'h', 'b']): axes[r, j].plot(TAU_FIT, [f[key] for f in fits], 'o-', ms=3, color=col, label=f'ep{ep}')
    for r, nm in enumerate(['a (bifurcation parameter)', 'h (asymmetry)', 'b (cubic stiffness)']):
        axes[r, j].set_xscale('log'); axes[r, j].axhline(0, color='k', lw=0.4); axes[r, j].set_title(f'{pair}: {nm}', fontsize=10); axes[r, j].legend(fontsize=7)
    axes[2, j].set_xlabel('tau')
plt.tight_layout(); plt.show()

print("mean |h_net - h_exact| over tau in [0.5, 1.2] (the window that matters):")
sel = (TAU_FIT > 0.5) & (TAU_FIT < 1.2)
for pair in PAIRS:
    he = np.array([e['h'] for e in COEF[(pair, 'exact')]])[sel]
    print(f"  {pair}: " + "  ".join(f"ep{ep}: {np.mean(np.abs(np.array([f['h'] for f in COEF[(pair, ep)]])[sel] - he)):.3f}" for ep in CKPTS)
          + f"   (signed mean h_net at ep80: {np.mean(np.array([f['h'] for f in COEF[(pair, 80)]])[sel]):+.3f}, exact {np.mean(he):+.3f})")

## 2. Sampling consequence: generated class fraction vs h_net

Deterministic Heun sampler (EDM, 40 steps, σ from 20 to 0.002). Class of each sample = sign of its LDA coordinate. Prediction: fraction of the `+` class (digit a) exceeds 0.5 when h_net > 0, and moves toward 0.5 as training proceeds.

In [ ]:
@torch.no_grad()
def sample(net, n=2000, steps=40, s_max=20.0, s_min=0.002, rho=7.0):
    i = torch.arange(steps, device=dev); sig = (s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho) - s_max**(1/rho)))**rho
    sig = torch.cat([sig, torch.zeros(1, device=dev)]); x = torch.randn(n, 1, 28, 28, device=dev)*sig[0]
    traj_m = []
    for k in range(steps):
        s0, s1 = sig[k], sig[k+1]; d0 = (x - denoise(net, x, s0.expand(n)))/s0; x1 = x + (s1-s0)*d0
        if s1 > 0:
            d1 = (x1 - denoise(net, x1, s1.expand(n)))/s1; x = x + (s1-s0)*0.5*(d0+d1)
        else: x = x1
        traj_m.append(x.view(n, -1).clone())
    return x.view(n, -1), sig[:-1].cpu().numpy(), traj_m

print(f"{'pair':>7} {'ep':>4} {'frac(+ class)':>14} {'h_net@0.8':>10} {'commit tau (median)':>20}   exact tau_bif~0.8")
RES = []
for pair in PAIRS:
    Xa, Xb, Xflat = balanced_flat(pair); u, mid, half = lda_direction(Xa, Xb)
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev)
    for ep in CKPTS:
        x, sigs, traj = sample(NETS[pair][ep])
        mfin = ((x - midt) @ ut / half).cpu().numpy(); frac = float((mfin > 0).mean())
        M = torch.stack([((t - midt) @ ut / half) for t in traj]).cpu().numpy()          # (steps, n)
        committed = np.abs(M) > 0.5; first = np.array([np.argmax(c) if c.any() else len(sigs)-1 for c in committed.T])
        tau_commit = (sigs[first]**2)/half**2
        hn = np.mean(np.array([f['h'] for f in COEF[(pair, ep)]])[(TAU_FIT > 0.5) & (TAU_FIT < 1.2)])
        RES.append((pair, ep, frac, hn, np.median(tau_commit)))
        print(f"{pair!s:>7} {ep:4d} {frac:14.3f} {hn:10.3f} {np.median(tau_commit):20.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
for pair, col in zip(PAIRS, ['C0', 'C1', 'C2']):
    r = [x for x in RES if x[0] == pair]; ax.plot([x[3] for x in r], [x[2] for x in r], 'o-', color=col, label=str(pair))
    for x in r: ax.annotate(f'ep{x[1]}', (x[3], x[2]), fontsize=7)
ax.axhline(0.5, color='k', ls='--', lw=0.8); ax.axvline(0, color='k', ls='--', lw=0.8)
ax.set_xlabel('h_net (mean over tau in [0.5,1.2])'); ax.set_ylabel('generated fraction of + class'); ax.legend(); ax.set_title('does h_net predict generated class bias?')
plt.tight_layout(); plt.show()

## What to report back

1. The coefficient figure and the |h_net − h_exact| table: convergence with training, per pair.
2. The sampling table and scatter: generated `+`-class fraction vs `h_net`. The prediction is a monotone relation crossing (0, 0.5). Also whether the median commitment τ of generated trajectories moves toward the exact fold τ with training (the delay effect seen from the sampler's side).